## Data Augmentation

In [ ]:
# DATA AUGMENTATION

ind = random.randint(0, np.max(np.shape(df_ESC50)))
path1 = df_ESC50.full_path[ind]
# audio = librosa.load(path1, sr = 44100) #instead of this
audio = tfio.audio.AudioIOTensor(
    path1
)  # AudioIOTensor is lazy-loaded so only shape, dtype, and sample rate are shown initially.
audio = tf.squeeze(audio[:], axis=[-1])
print(f"Category: {df_ESC50.category[ind]}\n")
display(ipd.Audio(data=audio.numpy(), rate=sample_rate))

print("Cast the tensor to new type")
audio = tf.cast(audio, tf.float32) / 32768.0
plt.subplots(6, 1, figsize=(9, 15), squeeze=False)
plt.tight_layout(pad=3)
plt.subplot(6, 1, 1)
plt.plot(audio.numpy())

print("Use tfio.audio.trim")
position = tfio.audio.trim(
    audio, axis=0, epsilon=0.1
)  # Returns a tensor of start and stop with shape [..., 2, ...].
print(f"The positions are {position}")

start = position[0]
stop = position[1]
print(start, stop)

processed = audio[
    start:stop
]  # we cannot modify thew shape of our  input...discard this method

plt.subplot(6, 1, 2)
plt.plot(processed.numpy())


fade = tfio.audio.fade(audio, fade_in=100000, fade_out=200000, mode="logarithmic")

plt.subplot(6, 1, 3)
plt.plot(fade.numpy())
display(ipd.Audio(data=fade.numpy(), rate=sample_rate))

# Convert to spectrogram
spectrogram = tfio.audio.spectrogram(audio, nfft=512, window=512, stride=256)

plt.subplot(6, 1, 4)
plt.imshow(tf.math.log(spectrogram).numpy(), aspect="auto")

# Convert to mel-spectrogram
mel_spectrogram = tfio.audio.melscale(
    spectrogram, rate=16000, mels=128, fmin=0, fmax=8000
)


plt.subplot(6, 1, 5)
plt.imshow(tf.math.log(mel_spectrogram).numpy(), aspect="auto")

# Convert to db scale mel-spectrogram
dbscale_mel_spectrogram = tfio.audio.dbscale(mel_spectrogram, top_db=80)

plt.subplot(6, 1, 6)
plt.imshow(dbscale_mel_spectrogram.numpy(), aspect="auto")

Here some methods for data augmentation.
Frequency and Time Masking discussed in SpecAugment: A Simple Data Augmentation Method for Automatic Speech Recognition (Park et al., 2019).


In [ ]:
plt.subplots(6, 1, figsize=(9, 15))
plt.tight_layout(pad=3)
for i in range(3):
    # Freq masking
    freq_mask = tfio.audio.freq_mask(dbscale_mel_spectrogram, param=20)

    plt.subplot(6, 1, 2 * i + 1)
    plt.title(f"freq mask number {i + 1}")
    plt.imshow(freq_mask.numpy(), aspect="auto")

    # Time masking
    time_mask = tfio.audio.time_mask(dbscale_mel_spectrogram, param=500)

    plt.subplot(6, 1, 2 * i + 2)
    plt.title(f"time mask number {i + 2}")
    plt.imshow(time_mask.numpy(), aspect="auto")

In [ ]:
audio_resample = tfio.audio.resample(input=audio, rate_in=sample_rate, rate_out=80000)
print(audio_resample.shape)

display(ipd.Audio(data=audio.numpy(), rate=sample_rate))
display(ipd.Audio(data=audio_resample.numpy(), rate=sample_rate))

plt.subplots(2, 1, figsize=(9, 5))
plt.tight_layout(pad=3)


plt.subplot(2, 1, 1)
plt.plot(audio.numpy())


plt.subplot(2, 1, 2)
plt.plot(audio_resample.numpy())

## Autoencoders


### Old code

#### Solved adjusting the kernel

In [ ]:
INPUT_DIM = (220500, 1)
code_size = 32
strides = 64


encoder = tf.keras.Sequential()
encoder.add(layers.InputLayer(input_shape=INPUT_DIM))
encoder.add(layers.Conv1D(16, 128, strides=strides, activation="relu", padding="same"))
encoder.add(layers.Conv1D(16, 128, strides=strides, activation="relu", padding="same"))
encoder.add(layers.Flatten())
encoder.add(layers.Dense(code_size, activation="relu"))

nodes = int(INPUT_DIM[0] / strides / strides * 16)
decoder = tf.keras.Sequential()
decoder.add(layers.InputLayer(input_shape=(code_size,)))
decoder.add(layers.Dense(nodes, activation="relu"))
decoder.add(layers.Reshape((nodes, 1)))
decoder.add(
    layers.Conv1DTranspose(16, 128, strides=256, activation="relu", padding="same")
)
decoder.add(
    layers.Conv1DTranspose(1, 85, strides=1, activation="linear", padding="valid")
)

encoder.summary()
decoder.summary()

# compile the model

encoder, decoder = build_autoencoder(INPUT_DIM, code_size=code_size)
# working with the autoencoder
inp = tf.keras.Input(INPUT_DIM)
code = encoder(inp)
reconstruction = decoder(code)
autoencoder = tf.keras.Model(inputs=inp, outputs=reconstruction)

# compile and fit
epochs = 1
autoencoder, history = compile_and_fit(
    autoencoder,
    train,
    val,
    epochs=epochs,
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=["mse"],
    verbose=1,
)

plot_history(history)

#### Solved second time

In [ ]:
import tensorflow as tf
from keras import layers

# INPUT_DIM = (220500,1)
code_size = 32
strides = 64
kernel_size = 128
n_filters = 16
kernel_size_T = INPUT_DIM[0] - strides * ((INPUT_DIM[0] - kernel_size) // strides)
print(kernel_size_T)


def build_autoencoder(INPUT_DIM, code_size=code_size):
    encoder = tf.keras.Sequential()
    encoder.add(layers.InputLayer(input_shape=INPUT_DIM))
    encoder.add(
        layers.Conv1D(
            n_filters, kernel_size, strides=strides, activation="relu", padding="valid"
        )
    )  # output_shape = (1, (INPUT_DIM-kernel_size)//strides+1, n_filters)
    # encoder.add(layers.Conv1D(n_filters, kernel_size, strides=strides, activation='relu', padding='valid'))
    encoder.add(
        layers.Flatten()
    )  # output_shape = (1, ((INPUT_DIM-kernel_size)//strides+1)* n_filters)
    encoder.add(layers.Dense(code_size, activation="relu"))

    r = ((INPUT_DIM[0] - kernel_size) // strides + 1) * n_filters
    decoder = tf.keras.Sequential()
    decoder.add(layers.InputLayer(input_shape=(code_size,)))
    decoder.add(layers.Dense(r, activation="relu"))
    decoder.add(
        layers.Reshape((((INPUT_DIM[0] - kernel_size) // strides + 1), n_filters))
    )
    # decoder.add(layers.Conv1DTranspose(n_filters, kernel_size, strides=256, activation='relu', padding='valid'))
    decoder.add(
        layers.Conv1DTranspose(
            1, kernel_size_T, strides=strides, activation="linear", padding="valid"
        )
    )  # output_shape = (1,strides*(input_shape-1)+kernel_size_T, 1 )

    return encoder, decoder


"""
import numpy as np
INPUT_DIM = 220500
strides =99
kernel_size = 345
n_filters = 16
print(layers.Conv1D(n_filters, kernel_size, strides=strides, activation='relu', padding='valid')(np.random.random((1, INPUT_DIM, 1))).shape)
output_shape = (1, (INPUT_DIM-kernel_size)//strides+1, n_filters)
print(output_shape)

input_shape = 22222
strides = 72
kernel_size = 2344
n_filters = 16
print(layers.Conv1DTranspose(n_filters, kernel_size, strides=strides, activation='relu', padding='valid')(np.random.random((1, input_shape , 1))).shape)
output_shape = (1,strides*(input_shape-1)+kernel_size,n_filters)
print(output_shape)
"""

# compile the model

encoder, decoder = build_autoencoder(INPUT_DIM, code_size=code_size)
encoder.summary()
decoder.summary()
# working with the autoencoder
inp = tf.keras.Input(INPUT_DIM)
code = encoder(inp)
reconstruction = decoder(code)
autoencoder = tf.keras.Model(inputs=inp, outputs=reconstruction)

# compile and fit
epochs = 1
autoencoder, history = compile_and_fit(
    autoencoder,
    train,
    val,
    epochs=epochs,
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=["mse"],
    verbose=1,
)

plot_history(history)

# evaluate the model

display(autoencoder.evaluate(test, return_dict=True))

#### 2nd Model

In [ ]:
def build_autoencoder(INPUT_DIM, code_size, strides=64, dropout=0.2):
    encoder = tf.keras.Sequential()
    encoder.add(layers.InputLayer(input_shape=INPUT_DIM))
    encoder.add(
        layers.Conv1D(64, 128, strides=strides, activation="relu", padding="same")
    )
    if dropout is not None:
        encoder.add(layers.Dropout(dropout))
    encoder.add(
        layers.Conv1D(64, 128, strides=strides, activation="relu", padding="same")
    )
    if dropout is not None:
        encoder.add(layers.Dropout(dropout))
    encoder.add(
        layers.Conv1D(32, 128, strides=strides, activation="relu", padding="same")
    )
    encoder.add(layers.Flatten())
    if dropout is not None:
        encoder.add(layers.Dropout(dropout))
    encoder.add(layers.Dense(code_size, activation="relu"))

    nodes = int(INPUT_DIM[0] / strides / strides * 16)
    decoder = tf.keras.Sequential()
    decoder.add(layers.InputLayer(input_shape=(code_size,)))
    decoder.add(layers.Dense(nodes, activation="relu"))
    decoder.add(layers.Reshape((nodes, 1)))
    if dropout is not None:
        decoder.add(layers.Dropout(dropout))
    decoder.add(
        layers.Conv1DTranspose(16, 128, strides=256, activation="relu", padding="valid")
    )
    if dropout is not None:
        decoder.add(layers.Dropout(dropout))
    decoder.add(
        layers.Conv1DTranspose(8, 128, strides=256, activation="relu", padding="valid")
    )
    if dropout is not None:
        decoder.add(layers.Dropout(dropout))
    decoder.add(
        layers.Conv1DTranspose(1, 128, strides=1, activation="linear", padding="valid")
    )
    desired_output_shape = (1, 220500, 1)
    output_cutter = OutputCutterLayer(desired_shape=desired_output_shape)
    decoder.add(output_cutter)
    if dropout is not None:
        decoder.add(layers.Dropout(dropout))

    return encoder, decoder

In [ ]:
# compile the model
code_size = 32
encoder, decoder = build_autoencoder(INPUT_DIM, code_size=code_size)

encoder.summary()
decoder.summary()

#### Other possible loss functions

In [ ]:
# interesting loss function fro images


def ssim_loss(y_true, y_pred):
    return 1 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, 2.0))


# optimizer = tf.train.AdamOptimizer(learning_rate).minimize(-1 * loss_rec)


# Load pre-trained VGG model with custom input shape
def create_vgg_model(input_shape=(None, None, 1)):
    vgg_model = tf.keras.applications.VGG16(
        include_top=False, weights="imagenet", input_shape=input_shape
    )

    # Remove the fully connected layers to make it a feature extraction model
    vgg_model = tf.keras.Model(
        inputs=vgg_model.input, outputs=vgg_model.get_layer("block4_conv3").output
    )

    # Freeze the model's layers to prevent updating during training
    vgg_model.trainable = False

    return vgg_model


# Assuming your input images have shape (height, width, 1)
input_shape = (INPUT_DIM[0], INPUT_DIM[1], 1)

# Create the custom VGG model
vgg_model = create_vgg_model(input_shape)


# Function to compute perceptual loss using mean squared error
def perceptual_loss(y_true, y_pred):
    true_features = vgg_model(y_true)
    pred_features = vgg_model(y_pred)
    return tf.keras.losses.MeanSquaredError()(true_features, pred_features)


vgg_model = tf.keras.applications.VGG16(
    include_top=False, weights="imagenet", input_shape=(INPUT_DIM[0], INPUT_DIM[1], 3)
)
vgg_model.trainable = False


def perceptual_loss(y_true, y_pred):
    def channel_3(a):
        return tf.convert_to_tensor([a, a, a]).transpose(axes=(1, 2, 3, 0))

    true_features = vgg_model(channel_3(y_true))
    pred_features = vgg_model(channel_3(y_pred))
    # print the shapes
    print(f"The shape of the true features is {true_features.shape}")
    print(f"The shape of the predicted features is {pred_features.shape}")

    return tf.keras.losses.MeanSquaredError()(true_features, pred_features)

In [ ]:
path_enc = os.path.join(main_dir, "Save_models", "encoder.keras")
encoder.save(path_enc, save_format="keras")

### Other autoencoder trials with attempt to use the encoder for classification

In [ ]:
# load the encoder
encoder_loaded = tf.keras.models.load_model(path_enc)

# show first element of the dataset
for example_train_batch, label in train.take(2):
    print(f"Audio shape: {example_train_batch.shape}")

    # print the types of object we have
    print(f"train batch type: {type(example_train_batch)}")

    # visualize the audio
    if len(example_train_batch[0].numpy().shape) < 3:
        plt.figure(figsize=(10, 4))
        plt.plot(example_train_batch[0].numpy())
        plt.title(f"Audio plot")
        # listen to the audio
        display(
            ipd.Audio(
                data=np.reshape(example_train_batch[0].numpy(), [-1]), rate=samplerate
            )
        )
    else:
        plt.figure(figsize=(10, 4))
        plt.imshow(example_train_batch[0].numpy(), aspect="auto")
        plt.colorbar()
        plt.title(f"{preprocessing} spectrogram")

    # predict the encoded vector
    y_pred = encoder_loaded.predict(example_train_batch)

    # show the encoded vector
    print(f"Encoded vector: {y_pred[0]}")
    print(f"Dim of encoded vector: {y_pred[0].shape}")
    print(label_names[np.argmax(label[0])])

# in train, val and test map the tuple (y,lab) to (y_pred,lab) where y_pred is the encoded vector and lab is the label creating new datasets
train_pred_labelled = train.map(lambda y, lab: (encoder_loaded(y), lab))
val_pred_labelled = val.map(lambda y, lab: (encoder_loaded(y), lab))
test_pred_labelled = test.map(lambda y, lab: (encoder_loaded(y), lab))

In [ ]:
# show the first element of train_labelled
for example_train_batch, label in train_pred_labelled.take(1):
    print(f"Encoded shape (batch, encoding dim): {example_train_batch.shape}")
    print(f"Encoded vector: {example_train_batch[0]}")
    print(f"Dim of encoded vector: {example_train_batch[0].shape}")
    print(f"Label of the encoded vector: {label[0]}")
    print(f"Label of the encoded vector: {label_names[np.argmax(label[0])]}")

### Training LSTM

In [ ]:
# exapand the dim of the encoded vector
train_pred_labelled_exp = train_pred_labelled.map(
    lambda y, lab: (tf.expand_dims(y, axis=-1), lab)
)
val_pred_labelled_exp = val_pred_labelled.map(
    lambda y, lab: (tf.expand_dims(y, axis=-1), lab)
)
test_pred_labelled_exp = test_pred_labelled.map(
    lambda y, lab: (tf.expand_dims(y, axis=-1), lab)
)

In [ ]:
from keras.layers import LSTM, Dense
from keras.models import Sequential

random_seed = 42

# define model LSTM sequential over train_pred_labelled dataset

model_LSTM = Sequential()
model_LSTM.add(LSTM(128, activation="tanh", input_shape=(32, 1)))
model_LSTM.add(Dense(10, activation="softmax"))
model_LSTM.summary()

# compile the model

model_LSTM.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=["accuracy"],
)

# fit the model
history = model_LSTM.fit(
    train_pred_labelled, epochs=500, validation_data=val_pred_labelled, verbose=1
)

# plot the history
plot_history(history)

# evaluate the model
print(f"Evaluate on test data")
display(model_LSTM.evaluate(test_pred_labelled, return_dict=True))

### LSTM more elegant

In [ ]:
# define a unique model with encoding layer and LSTM layer
layer_encoding = encoder_loaded
layer_encoding.trainable = False
model = tf.keras.models.Sequential(
    [
        tf.keras.layers.Input(shape=INPUT_DIM),
        layer_encoding,
        tf.keras.layers.Reshape((32, 1)),
        tf.keras.layers.LSTM(128, return_sequences=True),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.LSTM(128),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(10, activation="softmax"),
    ]
)

model.summary()

# shape of input (30,64,128,1)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=["accuracy"],
)

# fit the model
history = model.fit(train, epochs=200, validation_data=val, verbose=1)

# plot the history
plot_history(history)

# evaluate the model
print(f"Evaluate on test data")
display(model.evaluate(test, return_dict=True))

## Grid search over the preprocessing (inappropriate)

 this point we can implement a grid search to have a clear undestanding of which preprocessing performs better.

In [ ]:
ESC10_path = os.path.join(main_dir, "data", "ESC-10-depth")
preprocessing_type_list = [
    ["STFT"],
    ["MEL"],
    ["MFCC", False, False],
    ["MFCC", True, False],
    ["MFCC", True, True],
]
batch_size_list = [30, 90]
epochs = 100
patience = 10

best_model = None
best_accuracy = 0
for preprocessing_type in preprocessing_type_list:
    preprocessing = preprocessing_type[0]
    if preprocessing == "MFCC":
        delta = preprocessing_type[1]
        delta_delta = preprocessing_type[2]
    else:
        delta = False
        delta_delta = False
    print(f"Preprocessing type: {preprocessing}")

    for batch_size in batch_size_list:
        print(f"Batch size: {batch_size}")

        train, val, test, label_names = create_dataset(
            ESC10_path,
            batch_size=batch_size,
            preprocessing=preprocessing,
            delta=delta,
            delta_delta=delta_delta,
            normalize=True,
            verbose=0,
        )
        n_labels = len(label_names)
        for example_train_batch, label in train.take(1):
            INPUT_DIM = example_train_batch.shape[1:]
            plt.figure(figsize=(10, 4))
            plt.imshow(example_train_batch[0].numpy(), aspect="auto")
            plt.colorbar()
            plt.title(
                f"Preprocessing type: {preprocessing}, delta = {delta},  delta_delta = {delta_delta}"
            )

        # build the model

        model = tf.keras.models.Sequential(
            [
                tf.keras.layers.Conv2D(
                    16,
                    (3, 3),
                    strides=2,
                    activation="relu",
                    input_shape=INPUT_DIM,
                    padding="same",
                ),
                tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
                tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
                tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
                tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
                tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
                tf.keras.layers.Flatten(),
                tf.keras.layers.Dense(
                    n_labels, activation="softmax"
                ),  # Assuming 10 classes for classification
            ],
            name="CNN_" + preprocessing,
        )

        print("")
        print(
            f"CNN_{preprocessing},   batch_size = {batch_size},  delta = {delta},  delta_delta = {delta_delta}"
        )
        print("")

        model, hisotry, confusion_mtx, scores = compile_fit_evaluate(
            df_ESC10,
            model,
            train,
            val,
            test,
            label_names,
            epochs=epochs,
            patience=patience,
            loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            metrics=["accuracy"],  # ,'CategoricalAccuracy'],
            verbose=1,
            show_history=True,
            show_test_evaluation=True,
            show_confusion_matrix=True,
            listen_to_wrong=True,
        )
        if scores["accuracy"] > best_accuracy:
            best_model = model
            best_preprocessing = preprocessing
            best_delta = delta
            best_delta_delta = delta_delta
            best_accuracy = scores["accuracy"]

print(
    f"Best model: CNN_{best_preprocessing},   batch_size = {batch_size},  delta = {best_delta},  delta_delta = {best_delta_delta}"
)
print(f"Best score is {best_accuracy}")

In [ ]:
print(
    f"Best model: CNN_{best_preprocessing},   batch_size = {batch_size},  delta = {best_delta},  delta_delta = {best_delta_delta}"
)
print(f"Best score is {best_accuracy}")

#### Grid search over the Preprocessings (inappropriate)

At this point we can implement a grid search to have a clear undestanding of which preprocessing performs better.

In [ ]:
ESC50_path = os.path.join(main_dir, "data", "ESC-50-depth")
preprocessing_type_list = [
    ["STFT"],
    ["MEL"],
    ["MFCC", False, False],
    ["MFCC", True, False],
    ["MFCC", True, True],
]
batch_size_list = [60, 120]
epochs = 100
patience = 10

best_model = None
best_accuracy = 0
for preprocessing_type in preprocessing_type_list:
    preprocessing = preprocessing_type[0]
    if preprocessing == "MFCC":
        delta = preprocessing_type[1]
        delta_delta = preprocessing_type[2]
    else:
        delta = False
        delta_delta = False
    print(f"Preprocessing type: {preprocessing}")

    for batch_size in batch_size_list:
        print(f"Batch size: {batch_size}")

        train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
            ESC50_path,
            batch_size=batch_size,
            preprocessing=preprocessing,
            delta=delta,
            delta_delta=delta_delta,
            normalize=True,
            verbose=0,
            show_example_batch=True,
        )

        model = tf.keras.models.Sequential(
            [
                tf.keras.layers.Conv2D(
                    16,
                    (3, 3),
                    strides=2,
                    activation="relu",
                    input_shape=INPUT_DIM,
                    padding="same",
                ),
                tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
                tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
                tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
                tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
                tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),
                tf.keras.layers.Flatten(),
                tf.keras.layers.Dense(
                    n_labels, activation="softmax"
                ),  # Assuming 10 classes for classification
            ],
            name="CNN_" + preprocessing,
        )

        print("")
        print(
            f"CNN_{preprocessing},   batch_size = {batch_size},  delta = {delta},  delta_delta = {delta_delta}"
        )
        print("")

        model, hisotry, confusion_mtx, scores = compile_fit_evaluate(
            df_ESC10,
            model,
            train,
            val,
            test,
            label_names,
            epochs=epochs,
            patience=patience,
            loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
            metrics=["accuracy"],  # ,'CategoricalAccuracy'],
            verbose=1,
            show_history=True,
            show_test_evaluation=True,
            show_confusion_matrix=True,
            listen_to_wrong=False,
        )
        if scores["accuracy"] > best_accuracy:
            best_model = model
            best_preprocessing = preprocessing
            best_delta = delta
            best_delta_delta = delta_delta
            best_accuracy = scores["accuracy"]

print(
    f"Best model: CNN_{best_preprocessing},   batch_size = {batch_size},  delta = {best_delta},  delta_delta = {best_delta_delta}"
)
print(f"Best score is {best_accuracy}")

In [ ]:
print(
    f"Best model: CNN_{best_preprocessing},   batch_size = {batch_size},  delta = {best_delta},  delta_delta = {best_delta_delta}"
)
print(f"Best score is {best_accuracy}")

In [ ]:
best_model.save(
    os.path.join(main_dir, "Saved_Models", "best_ESC50_simple_CNN"), save_format="keras"
)

In [ ]:
model_loaded = tf.keras.models.load_model(
    os.path.join(main_dir, "Saved_Models", "best_ESC50_simple_CNN")
)
model_loaded.summary()

In [ ]:
model_loaded.predict(test)

In [ ]:
# The models are loaded with the weights corresponding to their best checkpoint (at the end of the best epoch of best trial).
# altrernatively
# best_score = tuner.load_model(tuner.oracle.get_best_trials(num_trials=1)[0]).evaluate(val, return_dict=True)['mse']

## Attention

### Preliminary cell to start the notebook

In [ ]:
# libraries
import os
import platform
import sys

print(sys.version)

strong_pc = platform.system() == "Linux"
in_colab = "google.colab" in sys.modules
if in_colab:
    if not os.getcwd().split("/")[-1].split("_")[-1] == "2023":
        from google.colab import drive

        drive.mount("/content/drive")
        os.chdir(r"/content/drive/MyDrive/Human_Data_Analytics_Project_2023")

    #!pip install tensorflow==2.11.0
    #!pip install tensorflow_text==2.11.0
    if not "tensorflow_io" in sys.modules:
        print("Installing tensorflow-IO")
        !pip install tensorflow-io
    if not "keras" in sys.modules and False:
        print("Installing keras")
        !pip install keras==2.11.0
    if not "scikeras" in sys.modules:
        print("Installing scikeras")
        !pip install scikeras[tensorflow]
    if not "keras-tuner" in sys.modules:
        print("installing keras tuner")
        !pip install keras-tuner
        !pip install numba==0.57.0


if "DEEPNOTE_ENV" in os.environ:
    os.chdir("/..")
    os.chdir("datasets")
    os.chdir("googledrivedeepnoteintegration")
    os.chdir("Human_Data_Analytics_Project_2023")
    if not "librosa" in sys.modules:
        print("Installing Librosa")
        !pip install librosa
    if not "scikeras" in sys.modules:
        print("Installing scikeras")
        !pip install scikeras[tensorflow]
    if not "keras-tuner" in sys.modules:
        print("installing keras tuner")
        !pip install keras-tuner
        !pip install numba==0.57.0

main_dir = os.getcwd()
if main_dir not in sys.path:
    print("Adding the folder for the modules")
    sys.path.append(main_dir)

import itertools
import json
import pickle
import random
import shutil
import subprocess
import time
import warnings

import h5py

# PLOT LIBRARIES
import matplotlib
import matplotlib.pyplot as plt

# BASE LIBRARIES
import numpy as np
import pandas as pd

%matplotlib inline
import IPython.display as ipd

# AUDIO LIBRARIES
import librosa
import tensorflow as tf
from keras.models import load_model
from scikeras.wrappers import KerasClassifier
from scipy import signal
from scipy.fft import fft, fftfreq, fftshift, ifft
from scipy.io import wavfile
from scipy.signal import periodogram, spectrogram, stft
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier

# MACHINE LEARNING LIBRARIES
from sklearn.model_selection import GridSearchCV, LeaveOneOut, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import check_random_state

# import plotly.express as px


# from pydub import AudioSegment


# GPU SETTINGS FOR LINUX and repressing warnings for windows. References for gpu: https://www.tensorflow.org/guide/gpu
show_gpu_activity = False
if sys.platform == "linux" and not in_colab:
    if show_gpu_activity:
        tf.debugging.set_log_device_placement(True)

    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        # Restrict TensorFlow to only allocate a part of memory on the first GPU
        try:
            tf.config.set_logical_device_configuration(
                gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=6800)]
            )
            logical_gpus = tf.config.list_logical_devices("GPU")
            print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
        except RuntimeError as e:
            # Virtual devices must be set before GPUs have been initialized
            print(e)
else:
    warnings.filterwarnings("ignore", category=UserWarning)

from keras import layers, models
from keras.utils import plot_model as tf_plot

if in_colab:
    import tensorflow_io as tfio
print("TensorFlow version:", tf.__version__)
# show keras version
import keras

print(f"keras version = {keras.__version__}")
import keras_tuner as kt

# import keras_tune as kt
from keras import layers
from keras.regularizers import L1L2
from tensorflow import keras

# kernel_regularizer=regularizers.L1L2(l1=1e-5, l2=1e-4) # we may use this in some layers...

# RANDOM SETTINGS
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)
check_random_state(seed)

# OUR PERSONAL FUNCTIONS
import importlib

from Models.basic_ml import (
    basic_ML_experiments,
    basic_ML_experiments_gridsearch,
    build_dataset,
    extract_flatten_MFCC,
)
from Preprocessing.data_loader import download_dataset, load_metadata
from Preprocessing.exploration_plots import (
    Spectral_Analysis,
    one_random_audio,
    plot_clip_overview,
)

# EVALUATION LIBRAIRES
from sklearn.metrics import (
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    auc,
    make_scorer,
    precision_recall_curve,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_curve,
)
from Visualization.model_plot import confusion_matrix, listen_to_wrong_audio

importlib.reload(importlib.import_module("Preprocessing.data_loader"))
importlib.reload(importlib.import_module("Models.basic_ml"))
importlib.reload(importlib.import_module("Visualization.model_plot"))

from Preprocessing.data_loader import load_metadata

df_ESC10, df_ESC50 = load_metadata(
    main_dir, heads=False, ESC_US=False, statistics=False
)

from Models.basic_ml import (
    basic_ML_experiments,
    basic_ML_experiments_gridsearch,
    build_dataset,
    extract_flatten_MFCC,
)
from Preprocessing.data_loader import load_metadata

importlib.reload(importlib.import_module("Models.ann_utils"))
importlib.reload(importlib.import_module("Visualization.model_plot"))

from Models.ann_utils import *
from Models.ann_utils import MFCCWithDeltaLayer, OutputCutterLayer
from Visualization.model_plot import (
    confusion_matrix,
    listen_to_wrong_audio,
    plot_history,
    visualize_the_weights,
)

ESC10_path = os.path.join(main_dir, "Data", "ESC-10-depth")
samplerate = 44100

### 2.3.4 CNN ATTENTION - STFT Preprocessed Audio

In [ ]:
importlib.reload(importlib.import_module("Models.ann_utils"))
from Models.ann_utils import (
    K_fold_training,
    compile_fit_evaluate,
    create_dataset,
    create_dataset_lite,
)

#### Create the dataset ESC-10

In [ ]:
batch_size = 30

dataset, label = create_dataset_lite(
    df_ESC10,
    batch_size=batch_size,
    preprocessing="STFT",
    ndim=3,
)

INPUT_DIM, n_labels = example_batch(dataset, label_names=list(label.columns))

#### Build the model

We are going to keep this function for sections 2.3.2, 2.3.3 and 2.3.4 to avoid repeating the code

In [ ]:
import os

import tensorflow as tf

# Define your self-attention layer separately
self_attention_layer = tf.keras.layers.Attention(use_scale=True)


def build_model(
    n_labels=n_labels,  # arguments to build the model
    INPUT_DIM=INPUT_DIM,
    n_units=8,
    kernel_size=(3, 3),
    activation="relu",
    # arguments to compile the model
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    learning_rate=1e-3,
    metrics=["accuracy"],
    verbose=0,
    compile=True,
):
    input_layer = tf.keras.layers.Input(shape=INPUT_DIM)

    # Convolutional layer 1
    x = tf.keras.layers.Conv2D(
        n_units, kernel_size, strides=2, activation=activation, padding="same"
    )(input_layer)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Continue with the rest of your model
    x = tf.keras.layers.Conv2D(
        2 * n_units, (3, 3), strides=2, activation=activation, padding="same"
    )(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)

    x = tf.keras.layers.Conv2D(
        4 * n_units, (3, 3), activation=activation, padding="same"
    )(x)
    x = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Apply self-attention layer 3
    x = self_attention_layer([x, x])

    x = tf.keras.layers.Flatten()(x)

    output_layer = tf.keras.layers.Dense(n_labels, activation="softmax")(x)

    model = tf.keras.Model(inputs=input_layer, outputs=output_layer, name="CNN")

    if compile:
        model.compile(
            loss=loss,
            optimizer=(
                tf.keras.optimizers.Adam(learning_rate=learning_rate)
                if sys.platform == "darwin"
                else tf.keras.optimizers.Adam(learning_rate=learning_rate)
            ),
            metrics=metrics,
        )
        if verbose > 0:
            model.summary()

    print(
        f"n_units {n_units}, activation {activation}, learning_rate {learning_rate}, kernel size {kernel_size}"
    )

    return model

#### Run a grid search to find the best params

In [ ]:
epochs = 50
patience = 10
params = {
    "INPUT_DIM": [INPUT_DIM],
    "n_units": [16, 32],
    "activation": ["relu", "tanh"],
    "learning_rate": [1e-3, 1e-4],
    "kernel_size": [(3, 3), (5, 5), (7, 7)],
}
K_fold = 4

model_cv, result, best_params = K_fold_training(
    dataset,
    build_model,
    params=params,
    epochs=epochs,
    patience=patience,
    verbose=0,
    K=K_fold,
)

In [ ]:
print(
    "The best params are:",
    {key: value for key, value in best_params.items() if key != "INPUT_DIM"},
)

# save the best_params in pickle
with open(
    os.path.join(main_dir, "Models", "best_params_CNN_STFT_231_Attention.pickle"), "wb"
) as handle:
    pickle.dump(best_params, handle, protocol=pickle.HIGHEST_PROTOCOL)

#### Train the best model on ESC-50

In [ ]:
# refit only the best model on ESC-50
seed = 42
tf.random.set_seed(seed)
ESC50_path = os.path.join(main_dir, "data", "ESC-50-depth")
batch_size = 30
preprocessing = "STFT"

train, val, test, label_names, INPUT_DIM, n_labels = create_dataset(
    ESC50_path,
    verbose=0,
    batch_size=batch_size,
    validation_split=0.25,  # this is the splitting of train vs validation + test
    normalize=True,  # normalization preprocessing (default is true)
    preprocessing=preprocessing,  # "STFT", "MEL", "MFCC" or None
    show_example_batch=True,
    ndim=3,
)

In [ ]:
# upload the best parameters
with open(
    os.path.join(main_dir, "Models", "best_params_CNN_STFT_231_Attention.pickle"), "rb"
) as handle:
    best_params = pickle.load(handle)

In [ ]:
# Build the model

model = build_model(n_labels=n_labels, compile=False, **best_params)

epochs = 100
patience = 10  # early stopping patience
lr = best_params["learning_rate"]
model, hisotry, confusion_mtx, evaluation = compile_fit_evaluate(
    df_ESC50,
    model,
    train,
    val,
    test,
    label_names,
    epochs=epochs,
    patience=patience,
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    optimizer=(
        tf.keras.optimizers.Adam(learning_rate=lr)
        if sys.platform == "darwin" or in_colab
        else tf.keras.optimizers.Adam(learning_rate=lr)
    ),
    metrics=["accuracy"],
    verbose=1,
    show_history=True,
    show_test_evaluation=True,
    show_confusion_matrix=True,
    listen_to_wrong=True,
)

In [ ]:
# show the weights of the first layer
visualize_the_weights(model, layer_number=1, n_filters=16, verbose=0)

In [ ]:
save_model_path = os.path.join(
    main_dir, "Saved_Models", "ESC50_simple_CNN_STFT_ATTENTION"
)
# save the model
model.save(save_model_path, save_format="keras")

## EX 3.2_1Dim_Conv_AE_raw_flatten

### Preliminary cell to start the notebook

In [ ]:
# libraries
import os
import platform
import sys

print(sys.version)

in_colab = "google.colab" in sys.modules
strong_pc = platform.system() == "Linux"

if in_colab:
    if not os.getcwd().split("/")[-1].split("_")[-1] == "2023":
        from google.colab import drive

        drive.mount("/content/drive")
        os.chdir(r"/content/drive/MyDrive/Human_Data_Analytics_Project_2023")

    if not "tensorflow_io" in sys.modules:
        print("Installing tensorflow-IO")
        !pip install tensorflow-io
    if not "keras" in sys.modules:
        print("Installing keras")
        !pip install keras==2.10.0
    if not "scikeras" in sys.modules:
        print("Installing scikeras")
        !pip install scikeras[tensorflow]
    if not "keras-tuner" in sys.modules:
        print("installing keras tuner")
        !pip install keras-tuner
        !pip install numba==0.57.0

main_dir = os.getcwd()
if main_dir not in sys.path:
    print("Adding the folder for the modules")
    sys.path.append(main_dir)

import itertools
import json
import pickle
import random
import shutil
import subprocess
import time
import warnings

import h5py

# PLOT LIBRARIES
import matplotlib
import matplotlib.pyplot as plt

# BASE LIBRARIES
import numpy as np
import pandas as pd

%matplotlib inline
import IPython.display as ipd

# AUDIO LIBRARIES
import librosa
import tensorflow as tf
from keras import layers, models
from keras.utils import plot_model as tf_plot
from scikeras.wrappers import KerasClassifier
from scipy import signal
from scipy.fft import fft, fftfreq, fftshift, ifft
from scipy.io import wavfile
from scipy.signal import periodogram, spectrogram, stft
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier

# MACHINE LEARNING LIBRARIES
from sklearn.model_selection import GridSearchCV, LeaveOneOut, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils import check_random_state

# import plotly.express as px


# from pydub import AudioSegment


if in_colab:
    import tensorflow_io as tfio
print("TensorFlow version:", tf.__version__)
# show keras version
import keras

print(f"keras version = {keras.__version__}")
import keras_tuner as kt

# import keras_tune as kt
from keras import layers
from keras.regularizers import L1L2
from tensorflow import keras

# kernel_regularizer=regularizers.L1L2(l1=1e-5, l2=1e-4) # we may use this in some layers...

# RANDOM SETTINGS
seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)
check_random_state(seed)

# OUR PERSONAL FUNCTIONS
import importlib

from Models.basic_ml import (
    basic_ML_experiments,
    basic_ML_experiments_gridsearch,
    build_dataset,
    extract_flatten_MFCC,
)
from Preprocessing.data_loader import download_dataset, load_metadata
from Preprocessing.exploration_plots import (
    Spectral_Analysis,
    one_random_audio,
    plot_clip_overview,
)

# EVALUATION LIBRAIRES
from sklearn.metrics import (
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    auc,
    make_scorer,
    precision_recall_curve,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_curve,
)
from Visualization.model_plot import confusion_matrix, listen_to_wrong_audio

importlib.reload(importlib.import_module("Preprocessing.data_loader"))
importlib.reload(importlib.import_module("Models.basic_ml"))
importlib.reload(importlib.import_module("Visualization.model_plot"))

from Preprocessing.data_loader import load_metadata

df_ESC10, df_ESC50 = load_metadata(
    main_dir, heads=False, ESC_US=False, statistics=False
)

from Models.basic_ml import (
    basic_ML_experiments,
    basic_ML_experiments_gridsearch,
    build_dataset,
    extract_flatten_MFCC,
)
from Preprocessing.data_loader import load_metadata

importlib.reload(importlib.import_module("Models.ann_utils"))
importlib.reload(importlib.import_module("Visualization.model_plot"))

from Models.ann_utils import *
from Models.ann_utils import MFCCWithDeltaLayer, OutputCutterLayer
from Visualization.model_plot import (
    confusion_matrix,
    listen_to_wrong_audio,
    plot_history,
    visualize_the_weights,
)

ESC10_path = os.path.join(main_dir, "data", "ESC-10-depth")
samplerate = 44100

### 3 UNSUPERVISED LEARNING: AUTOENCODERS

In [ ]:
import importlib

importlib.reload(importlib.import_module("Models.ann_utils"))
importlib.reload(importlib.import_module("Visualization.model_plot"))
importlib.reload(importlib.import_module("Preprocessing.data_loader"))
from Models.ann_utils import *
from Preprocessing.data_loader import reshape_US
from Visualization.model_plot import *

In [ ]:
folder_path = "Saved_Models"  # Replace this with the actual folder path
file_names = ["1Dim_Conv_AE_raw_flatten_count.txt"]

for name in file_names:
    file_path = os.path.join(main_dir, folder_path, name)
    with open(file_path, "w") as f:
        f.write("0")
    print(f"Created {name} with content '0' in folder {folder_path}")

#### 3.2 Autoencoder on raw audio - 1-dim convolution and flatten code

### Create the dataset

As before.

In [ ]:
folder_number = 1
batch_size = 30 if not strong_pc else 128
preprocessing = None
train, val, test, INPUT_DIM = create_US_dataset(
    preprocessing=preprocessing,
    folder_number=folder_number,
    batch_size=batch_size,
    main_dir=main_dir,
    ndim=2,
)

### Preparation to use Keras-Tuner

In [ ]:
INPUT_DIM = (220500, 1)
code_size = 32


def build_autoencoder(
    INPUT_DIM=INPUT_DIM,
    dense_layer_before_code=True,
    code_size=32,  # this is the number of channel in the code if dense_layer_before_code is False
    activation="tanh",
    n_layers=2,
    n_units=16,
    kernel_size=100,
    strides=25,
    max_pooling=2,
    regularizer=1e-4,
    batch_norm=False,
    drop_out=0.0,
    learning_rate=1e-3,
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=["mse"],
):
    encoder = tf.keras.Sequential(name="Encoder")
    encoder.add(layers.InputLayer(input_shape=INPUT_DIM))

    output_padding_list = []
    lambda_padding_list = []

    for i in range(n_layers):
        # define a gfrowing number of filters
        filters = (
            code_size
            if not dense_layer_before_code and i == n_layers - 1
            else (i + 1) * n_units
        )

        # stop the adding for loop if the kernel is too big for the length
        if i > 0 and (
            encoder.layers[-1].input_shape[1] < kernel_size
            or not (encoder.layers[-1].input_shape[1] - kernel_size) / strides > 1
        ):
            print(f"With this kernel and strides you can build at most {i} layers")
            n_layers = i
            break

        # add a 1d convolution
        encoder.add(
            layers.Conv1D(
                filters,
                kernel_size,
                strides=strides,
                activation=activation,
                padding="valid",
            )
        )

        # save the output padding for the decoder
        _, length, _ = encoder.layers[-1].input_shape
        pad = length - (((length - kernel_size) // strides) * strides + kernel_size)
        output_padding_list.append(pad)

        # add a max pooling layer
        encoder.add(layers.MaxPooling1D(max_pooling))

        # save the upsampling padding for the decoder
        _, length, _ = encoder.layers[-1].input_shape
        lambda_padding_list.append(length % max_pooling)

        # batch normalization and drop out
        if batch_norm:
            encoder.add(layers.BatchNormalization())
        if drop_out > 0:
            encoder.add(layers.Dropout(drop_out))

    # use a flatten code
    if dense_layer_before_code:
        encoder.add(layers.Flatten())
        encoder.add(layers.Dense(code_size, activation=activation))

    # decoder
    decoder = tf.keras.Sequential(name="Decoder")
    decoder.add(tf.keras.Input(shape=encoder.layers[-1].output_shape[1:]))

    # reshape the code to the output of the last layer of the encoder
    if dense_layer_before_code:
        last_shape = encoder.layers[-3].output_shape[1:]
        decoder.add(layers.Dense(np.prod(last_shape), activation=activation))
        decoder.add(layers.Reshape(last_shape))

    # transpose convolutions
    for i in range(n_layers):
        # upsampling to revert max pooling
        decoder.add(layers.UpSampling1D(size=max_pooling))

        # pad to the correct shape before the maxpooling
        pad = lambda_padding_list[-i - 1]
        decoder.add(layers.Lambda(lambda x: tf.pad(x, [[0, 0], [0, pad], [0, 0]])))

        # define a decreasing number of kernel in the transpose convolution
        filters = n_units * (n_layers - i - 1) if i < n_layers - 1 else INPUT_DIM[-1]

        # transpose convolution with the correct padding
        pad = output_padding_list[-i - 1]
        decoder.add(
            layers.Conv1DTranspose(
                filters,
                kernel_size,
                strides=strides,
                activation=activation,
                output_padding=pad,
            )
        )

        # batch normalization and drop out
        if batch_norm:
            decoder.add(layers.BatchNormalization())
        if drop_out > 0:
            decoder.add(layers.Dropout(drop_out))

    # reshape the output to the input shape if needed
    if decoder.layers[-1].output_shape[1:] != INPUT_DIM:
        decoder.add(
            tf.keras.layers.Resizing(
                height=img_shape[0],
                width=img_shape[1],
                interpolation="bilinear",
                crop_to_aspect_ratio=False,
            )
        )

    # build the autoencoder with keras.Model
    inp = tf.keras.Input(shape=INPUT_DIM)
    code = encoder(inp)
    reconstruction = decoder(code)
    autoencoder = tf.keras.Model(
        inputs=inp, outputs=reconstruction, name="1Dim_Conv_AE_raw_flatten"
    )

    # compile the autoencoder
    lr = learning_rate
    optimizer = (
        tf.keras.optimizers.Adam(learning_rate=lr)
        if sys.platform == "darwin" or in_colab
        else tf.keras.optimizers.Adam(learning_rate=lr)
    )
    autoencoder.compile(optimizer=optimizer, loss=loss, metrics=metrics)

    # print the number of trainable parameters
    print(
        f"Model built with {sum(tf.keras.backend.count_params(p) for p in autoencoder.trainable_variables)} trainable params"
    )

    return autoencoder

In [ ]:
verbose = 2
# test the build_autoencoder function
autoencoder = build_autoencoder(
    n_layers=3, kernel_size=100, strides=50, max_pooling=2, n_units=16, code_size=32
)
if verbose > 1:
    autoencoder.summary(line_length=100)
    autoencoder.layers[1].summary(line_length=100)
    autoencoder.layers[2].summary(line_length=100)

In [ ]:
# function to build the model using different hyperparameters (keras tuner code)


def build_model(hp, test=False):
    # define the hyperparameters
    if test:
        print("Running a smaller grid search as a test")
        # dense_layer_before_code = False
        code_size = hp.Choice(name="code_size", values=[32, 64])
        activation = "tanh"
        n_layers = 1
        n_units = 16
        kernel_size = 100
        strides = 25
        max_pooling = 2
        dropout = 0.0
        batch_norm = hp.Boolean("batch_norm", default=True)
        learning_rate = 1e-3
    else:
        # dense_layer_before_code = hp.Boolean('dense_layer_before_code', default=True)
        code_size = hp.Choice(name="code_size", values=[8, 16, 32], default=32)
        activation = hp.Choice(
            name="activation", values=["relu", "tanh", "elu"], default="tanh"
        )
        n_layers = hp.Choice(name="n_layers", values=[1, 2, 3, 4], default=1)
        n_units = hp.Choice(name="n_units", values=[8, 16, 32, 64], default=16)
        kernel_size = hp.Choice(name="kernel_size", values=[50, 100, 150], default=100)
        strides = hp.Choice(name="strides", values=[10, 25, 50], default=25)
        max_pooling = hp.Choice(name="max_pooling", values=[2, 4, 6], default=2)
        batch_norm = hp.Boolean("batch_norm", default=True)
        dropout = hp.Choice(name="dropout", values=[0.0, 0.2, 0.5], default=0.0)
        learning_rate = hp.Choice(
            "learning_rate",
            values=[1e-4, 1e-3, 5 * 1e-3, 1e-2, 5 * 1e-2, 1e-1],
            default=1e-3,
        )

    dense_layer_before_code = True
    # define the model
    model = build_autoencoder(
        dense_layer_before_code=dense_layer_before_code,
        code_size=code_size,
        activation=activation,
        n_layers=n_layers,
        n_units=n_units,
        kernel_size=kernel_size,
        strides=strides,
        max_pooling=max_pooling,
        batch_norm=batch_norm,
        drop_out=dropout,
        learning_rate=learning_rate,
    )

    return model

In [ ]:
# test the build_model function
build_model(kt.HyperParameters()).summary()

### Implement the grid search hyperparameter-wise

In [ ]:
# dictionary with the default values of the hyperparams to be update each time
default_values = dict(
    code_size=32,
    activation="tanh",
    n_layers=1,
    n_units=16,
    kernel_size=100,
    strides=25,
    max_pooling=2,
    batch_norm=False,
    dropout=0.0,
    learning_rate=1e-3,
)

key_list = list(default_values.keys())

# define the general variables for our tuner
hpo_methods = ["RandomSearch", "BayesianOptimization", "Hyperband"]
max_model_size = 10**7
max_trials = 100
dir_name = os.path.join(main_dir, "1Dim_Conv_AE_raw_flatten")
verbose = 0

# define a smaller dataset for the grid search
if strong_pc:
    train_small = train
    val_small = val
else:
    small_size_dataset = 400
    train_val_small = train.unbatch().take(small_size_dataset)
    train_small = train_val_small.skip(100).batch(25)
    val_small = train_val_small.take(100).batch(25)

# define a list to collect all the best scores
best_score_dict = {
    "RandomSearch": [],
    "BayesianOptimization": [],
    "Hyperband": [],
}  # we hope to have a decreasing list of numbers...

# to be consistent with this type of grd search we should pass each hp more than one time...
for hpo_method in hpo_methods:
    random.shuffle(key_list)
    for hyper_params in [key_list[0]]:  # key_list:
        print(f"Searching for the best value for {hyper_params}")

        # define an hp set with all fix but one
        hp = kt.HyperParameters()

        for fixed_param in default_values.keys():
            if fixed_param != hyper_params:
                hp.Fixed(name=fixed_param, value=default_values[fixed_param])

        if verbose > 1:
            display(hp.space)

        # create a tuner for the params not fixed
        tuner = build_tuner(
            build_model=build_model,
            hpo_method=hpo_method,
            max_model_size=max_model_size,
            max_trials=max_trials,
            dir_name=dir_name,
            overwrite=True,
            objective=kt.Objective("val_mse", direction="min"),
            hp=hp,
            not_fixed_param=hyper_params,
            tune_new_entries=True,
        )

        if verbose > 2:
            display(tuner.search_space_summary(extended=True))

        # fit the tuner
        epochs = 1 if not strong_pc else 50
        patience = 10
        metrics = ["mse"]
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_" + metrics[0], verbose=verbose, patience=patience
            )
        ]

        tuner.search(
            train_small,
            validation_data=val_small,
            callbacks=callbacks,
            epochs=epochs,
            verbose=int(verbose > 0),
        )

        # retrive the best value for the free hp
        best_value = tuner.get_best_hyperparameters()[0].values[hyper_params]

        try:
            # retrive the best score reached
            best_score = tuner.get_best_models(num_models=1)[0].evaluate(
                val, return_dict=True
            )["mse"]
            print(
                f"The best value for {hyper_params} is {best_value}, the best score is {best_score}"
            )
            best_score_dict[hpo_method].append(best_score)
        except:
            print("For this trial we can not compute the best score")

        # update the default dict of values
        default_values[hyper_params] = best_value

        # save the updated dictionary
        file_path = os.path.join(main_dir, dir_name, hpo_method + "_best_params")
        with open(file_path, "wb") as file:
            pickle.dump(default_values, file)

        # delete the folder just created by the run
        shutil.rmtree(os.path.join(main_dir, dir_name, hpo_method + "_" + hyper_params))

    with open(file_path, "rb") as file:
        best_params = pickle.load(file)

    display(best_params)


# save the best_score_dict
file_path = os.path.join(main_dir, dir_name, "best_scores")
with open(file_path, "wb") as file:
    pickle.dump(best_score_dict, file)

with open(file_path, "rb") as file:
    best_scores = pickle.load(file)

display(best_scores)

In [ ]:
# compare the best hp from the 3 grid search methods
hyperparamters = []
for hpo_method in ["RandomSearch", "BayesianOptimization", "Hyperband"]:
    file_path = os.path.join(main_dir, dir_name, hpo_method + "_best_params")
    with open(file_path, "rb") as file:
        hyperparamters.append(pickle.load(file))
pd.DataFrame(
    hyperparamters, index=["RandomSearch", "BayesianOptimization", "Hyperband"]
)

### Train the model with best params with more data

In [ ]:
best_params = default_params  # DA METTERE A MANO I VERI BERST PARAMS

# build an autoencoder with the best params
autoencoder = build_autoencoder(**best_params)

# autoencoder = tuner.get_best_models(num_models=1)[0] #to create the model with some already wuite good weights
autoencoder.summary()
verbose = 0
if verbose > 0:
    autoencoder.layers[1].summary()
    autoencoder.layers[2].summary()

US_training(
    AE_name="1Dim_Conv_AE_raw_flatten", autoencoder=autoencoder, epochs=1, n_folders=2
)

### Show the reconstruction capabilities of the model

In [ ]:
# load the saved model
model_loaded = tf.keras.models.load_model(
    os.path.join(main_dir, "Saved_Models", "1Dim_Conv_AE_raw_flatten")
)
model_loaded.summary()

# plot the original and reconstructed
plot_original_reconstructed_raw(model=model_loaded, n_figures=5, test=test)